# The Richards Equation
_1-D unsaturated water flow_  
  
Caoimhe Mannion, November 2026  


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.interpolate import RectBivariateSpline as rbs
import sympy as sp

#### Parameters

In [ ]:
# example values, to update 

# theta values, fraction in [0,1], upper limit set by soil porosity
theta_r = 0.1 #residual volumetric water content as K -> 0
theta_s = 0.5 #volumetric water content at saturation

lambda_ = 0.9 #pore size distribution index
#K_s = 0.01 #cm/s, Saturation Conductivity

psi_b = None #matric potential
h_b = 20 #cm, matric potential head

In [ ]:
# Values for Touchet Silt Loam, Brooks & Corey 1964
porosity = 0.485 #fraction
h_b = 75 #cm
k_perm = 0.6 # intrinsic permeability of the soil, sq microns
lambda_ = 1.82
S_r = 0.270 #residual saturation (vwc of pore space)

In [22]:
# Get required parameters
k_perm = k_perm*10**(-8) #unit conversion to sq cm
rho = 1 # density of water, g/cm^3
g = 9.81 #acceleration due to gravity
mu = 0.01002 #dynamic viscosity of water at 20C, g/cm*s
K_s = k_perm*rho*g/mu
print(f"K_s = {K_s:.3e}")

theta_r = S_r*porosity
theta_s = porosity
print(f"theta_r, theta_s = {theta_r:.3f}, {theta_s:.3f}")

K_s = 5.874e-46
theta_r, theta_s = 0.131, 0.485


#### Conditions
$\theta(t, z)$

In [ ]:
# initial condition
#theta_t0 = some function
T = 300 #s e.g.
t_range = np.linspace(0,T,T+1) 

# boundary conditions: edges of range
theta_z0 = theta_r
theta_zd = theta_s
d = 100 #cm, depth in soil
z_range = np.linspace(0,d,d+1)

#### Functions
##### Brooks-Corey Retention Curve  
$$ S_e = \frac{\theta-\theta_r}{\theta_s-\theta_r} = \left(\frac{\psi_b}{\psi_m}\right)^{\lambda} = \left(\frac{h_b}{|h|}\right)^{\lambda} $$
For $\psi_b > \psi_m$ and $\psi_b, \psi_m$ same units and sign  
$h_b > h$ and $h_b, h$ same units (length). $h$ is negative in the unsaturated zone, $h_b$ is a positive constant, characteristic of the soil.
  
If $\psi_b \leq \psi_m$ or $h_b \leq h$, $S_e = 1$
  
##### Hydraulic Conductivity
$$ K\left(\theta \right)=K_{s}{\left(S_{e}\right)}^{n} $$
  
##### Water Diffusivity Function
$$ D_{w}\left(\theta \right)=K\left(\theta \right)\frac{dh}{d\theta }=K\left(\theta \right)\frac{h_{b}}{\lambda \left({\theta }_{s}-{\theta }_{r}\right)}S_{e}^{-1/\lambda -1}$$

### Mixed Form
$$\frac{\partial \theta}{\partial t}=\frac{\partial}{\partial z}\left[K(h)\left(\frac{\partial h}{\partial z}+1 \right)\right]

In [ ]:
### Mixed Form ### 
h = np.linspace(-10e-5,0,30) #matric potential head, centimetres, negative for unsaturated soil, 0 at saturation.

def S_e(h):
    """Brooks-Corey Effective Saturation"""
    if abs(h) >= h_b:
        # saturation
        return 1
    else:
        return (h_b/abs(h))**(lambda_)

def theta(h):
    """Water Retention Curve"""
    return S_e(h)*(theta_s - theta_r) + theta_r

def K(h):
    """Burdine Hydraulic Conductivity"""
    return K_s*S_e(h)**(3+2/lambda_)

In [7]:
def rich_mxd(h):
    return sp.diff(K(h)*(sp.diff(h,z)+1),z)

### Water Content Form
$$\frac{\partial \theta}{\partial t}=\frac{\partial}{\partial z}\left[D_w(\theta)\left(\frac{\partial \theta}{\partial z}\right)+\frac{\partial K(\theta)}{\partial z}\right]

In [23]:
### Water Content Form ###
theta_arr = np.linspace(theta_r,theta_s,101) #volumetric water content, dimensionless, defined strictly between theta_r,theta_s (functions are for unsaturated soil)

def K(theta):
    """Burdine Hydraulic Conductivity"""
    return K_s*S_e(theta)**(3+2/lambda_)

def S_e(theta):
    """Brooks-Corey Retention Curve"""
    return (theta-theta_r)/(theta_s-theta_r)

def D(theta):
    """Water Diffusivity Function"""
    return K(theta)*h_b/(lambda_(theta_s-theta_r))*S_e(theta)**(-1/lambda_ - 1)